In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Final Project Submission (Labs 9–12)\n",
    "## Labs 9–11: Data Exploration, Preprocessing, and Modeling\n",
    "---\n",
    "### Lab 9 – Data Loading & Exploration"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "%matplotlib inline\n",
    "\n",
    "df = pd.read_csv('healthcare_dataset.csv')\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Basic info\n",
    "print(\"Shape:\", df.shape)\n",
    "df.info()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Statistical summary\n",
    "df.describe()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Check missing values\n",
    "df.isnull().sum()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Value counts for categorical columns\n",
    "for col in ['Gender', 'Diagnosis', 'Treatment_Plan']:\n",
    "    print(f\"\\n{col} distribution:\\n{df[col].value_counts()}\\n\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Histograms\n",
    "df[['Age','Blood_Pressure','Heart_Rate','Cholesterol_Level','BMI']].hist(figsize=(12,8), bins=20)\n",
    "plt.suptitle('Histograms of Numerical Features')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Count plots\n",
    "plt.figure(figsize=(6,4))\n",
    "sns.countplot(data=df, x='Diagnosis')\n",
    "plt.xticks(rotation=45)\n",
    "plt.title('Diagnosis Count')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Correlation heatmap\n",
    "corr = df[['Age','Blood_Pressure','Heart_Rate','Cholesterol_Level','BMI']].corr()\n",
    "sns.heatmap(corr, annot=True, cmap='coolwarm')\n",
    "plt.title('Correlation Matrix')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Lab 10 – Data Pre‑Processing\n",
    "* Dealing with Null Values (none found, but demonstrated)\n",
    "* Changing Datatypes to Int"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# If there were nulls, we would do: df.fillna(...) or df.dropna()\n",
    "# For demonstration, we just confirm no nulls.\n",
    "print(\"Missing values:\", df.isnull().sum().sum())\n",
    "\n",
    "# Convert columns to int\n",
    "int_cols = ['Patient_ID','Age','Blood_Pressure','Heart_Rate','Cholesterol_Level','BMI']\n",
    "for col in int_cols:\n",
    "    df[col] = df[col].astype(int)\n",
    "\n",
    "# Convert date\n",
    "df['Follow_Up_Date'] = pd.to_datetime(df['Follow_Up_Date'])\n",
    "print(\"Data types after conversion:\")\n",
    "print(df.dtypes)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Lab 11 – Model Training & Evaluation\n",
    "* Train/Test Split\n",
    "* Choose Random Forest Classifier\n",
    "* Test / Predict\n",
    "* Display Accuracy"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.preprocessing import LabelEncoder\n",
    "from sklearn.ensemble import RandomForestClassifier\n",
    "from sklearn.metrics import accuracy_score, classification_report\n",
    "\n",
    "# Prepare features (drop leakage columns)\n",
    "X = df.drop(columns=['Patient_ID','Follow_Up_Date','Treatment_Plan','Diagnosis'])\n",
    "y = df['Diagnosis']\n",
    "\n",
    "# Encode gender and target\n",
    "gender_enc = LabelEncoder()\n",
    "X['Gender'] = gender_enc.fit_transform(X['Gender'])\n",
    "target_enc = LabelEncoder()\n",
    "y_enc = target_enc.fit_transform(y)\n",
    "\n",
    "# Split\n",
    "X_train, X_test, y_train, y_test = train_test_split(\n",
    "    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc)\n",
    "print(f\"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Train Random Forest\n",
    "model = RandomForestClassifier(n_estimators=100, random_state=42)\n",
    "model.fit(X_train, y_train)\n",
    "\n",
    "# Predict & evaluate\n",
    "y_pred = model.predict(X_test)\n",
    "acc = accuracy_score(y_test, y_pred)\n",
    "print(f\"Accuracy: {acc:.4f}\")\n",
    "print(\"\\nClassification Report:\")\n",
    "print(classification_report(y_test, y_pred, target_names=target_enc.classes_))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Model Persistence for Flask (Lab 12)\n",
    "Save model and encoders for the web app."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pickle\n",
    "pickle.dump(model, open('diagnosis_model.pkl', 'wb'))\n",
    "pickle.dump(gender_enc, open('gender_encoder.pkl', 'wb'))\n",
    "pickle.dump(target_enc, open('target_encoder.pkl', 'wb'))\n",
    "print(\"Model and encoders saved.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## End of Labs 9–11 Notebook\n",
    "---\n",
    "Now move to `app.py` and `templates/index.html` for the Flask web application (Lab 12)."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.8"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}